# 小型企業 ERP 資料 — 商業分析綜合版

**情境**：一間台灣中小型車用百貨的 Access ERP 系統備份，經去識別化後做綜合商業分析。

**目標**：用一份 Notebook 回答老闆 5 個最重要的問題：
1. 這些年我們到底賺多少？毛利率走勢如何？
2. 多少錢死在倉庫裡？哪些商品該處理？
3. 誰是真金主？80% 營收來自幾位客戶？
4. 我們有多少帳還沒收回來？誰最可能變壞帳？
5. 下個月預計營收多少？哪些月份表現特別異常？

**資料**：`db1_backup_20260414_csv_anonymized/`（30+ 匿名 CSV 表，2020–2026 共 77 個月）

**方法框架**：Ask → Prepare → Process → Analyze → Share → Act（Google Analytics 方法論）

## 0. 環境準備

In [ ]:
import sys
from pathlib import Path

HERE = Path.cwd()
WORK = HERE.parent if HERE.name == 'notebooks' else HERE
sys.path.insert(0, str(WORK))

import pandas as pd
import matplotlib.pyplot as plt

from src.loaders import load_all
from src.enrich import enrich_sales_lines, enrich_purchase_lines

CSV_DIR = WORK / 'db1_backup_20260414_csv_anonymized'
data = load_all(CSV_DIR)
print(f'參考截止日：{data.ref_date.date()}')
print(f'客戶 {len(data.cust):,}｜商品 {len(data.prod):,}｜廠商 {len(data.fact):,}')
print(f'銷貨 {len(data.sale):,} 單 / {len(data.sales1):,} 行｜進貨 {len(data.purc):,} 單 / {len(data.purcs1):,} 行')

## 1. 我們到底賺多少？— 營收與毛利

In [ ]:
lines = enrich_sales_lines(data)
monthly = lines.dropna(subset=['month']).groupby('month', as_index=False).agg(
    revenue=('line_revenue', 'sum'),
    margin=('line_margin', 'sum'),
)
monthly['margin_rate'] = monthly['margin'] / monthly['revenue']
total_rev = monthly['revenue'].sum()
total_mar = monthly['margin'].sum()
print(f'累計營收：{total_rev:,.0f}')
print(f'累計毛利：{total_mar:,.0f}（毛利率 {total_mar/total_rev*100:.1f}%）')

fig, ax1 = plt.subplots(figsize=(12, 4.5))
ax1.bar(monthly['month'], monthly['revenue'], width=20, alpha=0.35, label='營收')
ax1.bar(monthly['month'], monthly['margin'], width=20, alpha=0.8, label='毛利')
ax2 = ax1.twinx()
ax2.plot(monthly['month'], monthly['margin_rate']*100, color='darkred', marker='o', linewidth=1.5, label='毛利率(%)')
ax1.set_title('月營收 vs 毛利 與毛利率')
ax1.legend(loc='upper left'); ax2.legend(loc='upper right')
ax1.grid(True, alpha=0.3)
plt.show()

**觀察**：整體毛利率維持在 50% 上下，但早期（2020–2021）與近期有差異，可由下方 YoY 圖進一步檢視。

## 2. 多少錢死在倉庫裡？— 呆料金額化 + ABC 分析

In [ ]:
import numpy as np
prod = data.prod.copy()
ref = data.ref_date
prod['last_activity'] = prod[['lastin','lastout']].max(axis=1)
prod['idle_days'] = (ref - prod['last_activity']).dt.days
stale = prod[(prod['lastin'].notna() | prod['lastout'].notna()) & (prod['idle_days'] > 365)].copy()
stale['dead_value'] = stale['qty'] * stale['pcost']
stale = stale.sort_values('dead_value', ascending=False)

print(f'呆料品項數：{len(stale):,}')
print(f'呆料金額合計：{stale["dead_value"].sum():,.0f}')

# 呆料金額前 15
stale[['prdnm','qty','pcost','dead_value','idle_days']].head(15)

In [ ]:
# ABC 分析
by_prod = lines.groupby('prdno', as_index=False).agg(revenue=('line_revenue','sum'))
by_prod = by_prod.sort_values('revenue', ascending=False).reset_index(drop=True)
total = by_prod['revenue'].sum()
by_prod['cum_share'] = by_prod['revenue'].cumsum() / total
by_prod['abc'] = np.where(by_prod['cum_share']<=0.8,'A', np.where(by_prod['cum_share']<=0.95,'B','C'))
counts = by_prod['abc'].value_counts().reindex(['A','B','C'])
share  = by_prod.groupby('abc')['revenue'].sum() / total
print('ABC 結構：')
for c in ['A','B','C']:
    print(f'  {c} 類 — {int(counts[c]):,} 品項 ({counts[c]/len(by_prod)*100:.1f}%)｜營收占比 {share[c]*100:.1f}%')

fig, ax1 = plt.subplots(figsize=(11, 4))
ax1.bar(range(len(by_prod)), by_prod['revenue'], alpha=0.5)
ax2 = ax1.twinx()
ax2.plot(range(len(by_prod)), by_prod['cum_share']*100, color='darkred')
ax2.axhline(80, color='grey', linestyle='--'); ax2.axhline(95, color='grey', linestyle=':')
ax1.set_title('商品 ABC Pareto')
ax1.set_ylabel('營收'); ax2.set_ylabel('累計占比 (%)')
plt.show()

**行動建議**：
- 呆料 Top 15 先做促銷清倉（標原價 5–6 折）或整批退廠
- C 類（營收只佔 5%，卻佔品項數一大半）考慮停止進貨，縮減 SKU 管理成本

## 3. 誰是真金主？— RFM + 80/20

In [ ]:
cus = lines.groupby('cusno', as_index=False).agg(
    last_purchase=('sdate','max'),
    frequency=('salno','nunique'),
    monetary=('line_revenue','sum'),
    margin=('line_margin','sum'),
)
cus['recency_days'] = (ref - cus['last_purchase']).dt.days
cus = cus.sort_values('monetary', ascending=False).reset_index(drop=True)
cus['cum_share'] = cus['monetary'].cumsum() / cus['monetary'].sum()
n_80 = (cus['cum_share'] < 0.8).sum() + 1
print(f'總客戶 {len(cus)}｜前 {n_80} 位（{n_80/len(cus)*100:.1f}%）貢獻 80% 營收')

fig, ax1 = plt.subplots(figsize=(11,4))
ax1.bar(range(len(cus)), cus['monetary'], alpha=0.5)
ax2 = ax1.twinx(); ax2.plot(range(len(cus)), cus['cum_share']*100, color='darkred'); ax2.axhline(80, color='grey', linestyle='--')
ax1.set_title('客戶營收 Pareto (80/20)')
plt.show()

## 4. 我們還有多少帳沒收？— AR Aging

In [ ]:
from src.analyses import cashflow as m_cf
# 直接呼叫模組計算帳齡
out = Path('/tmp/_bvnb_out'); out.mkdir(exist_ok=True)
res = m_cf.run(data, out)
for k, v in res.kpis.items():
    print(f'{k}：{v}')

**警示**：若 DSO（Days Sales Outstanding，收款週期）遠大於約定的 settle 天數，代表收款效率差，需要立刻啟動催收流程。

## 5. 下個月會怎樣？— 預測與異常月份

In [ ]:
from src.analyses import forecast as m_fc
res = m_fc.run(data, out)
for k, v in res.kpis.items():
    print(f'{k}：{v}')
# 圖表已於 analysis_output/report.html 內呈現

## 6. 總結 & 行動建議（Act）

| # | 發現 | 建議行動 |
|---|------|----------|
| 1 | 呆料金額顯著 | Top N 呆料品項優先清倉（特價／退廠） |
| 2 | 約 1/3 客戶貢獻 80% 營收 | 建立 VIP 名單，優先維繫；對 Champions 做 loyalty program |
| 3 | 壞帳風險清單 | 逾 1 年未交易且有未收 → 立即發送催繳／法務評估 |
| 4 | 商品關聯 | Top 20 商品對套用「買 A 推 B」搭售促銷 |
| 5 | 供應商集中 | 前 5 大廠商佔近半採購 → 評估備援廠商 |
| 6 | 負毛利商品 | 檢討定價 or 停售 |

**完整互動版**：`streamlit run dashboard.py`
**離線瀏覽**：雙擊開啟 `analysis_output/report.html`